# 节点 08 — Word2Vec：让词找到它的「邻居」

**目标**：
1. 理解 one-hot 编码的缺陷，感受「词向量」解决了什么
2. 手撕 Skip-gram 的前向传播和负采样目标函数
3. 用梯度下降训练词向量，验证语义相近的词在向量空间中靠得更近

**前置知识**：矩阵乘法（初中水平），Python 列表/字典，NumPy 基础

In [ ]:
import numpy as np
np.random.seed(42)

## 1. 一个玩具语料

我们用 8 个句子、10 个词的小语料来演示。
词典里，动物词（猫/狗/鸟）和食物词（鱼/肉/米）各自经常出现在一起。

In [ ]:
sentences = [
    ["猫", "吃", "鱼"],
    ["狗", "吃", "肉"],
    ["鸟", "吃", "米"],
    ["猫", "喜欢", "鱼"],
    ["狗", "喜欢", "肉"],
    ["鸟", "喜欢", "米"],
    ["猫", "狗", "鸟"],
    ["鱼", "肉", "米"],
]

# 构建词典（词 → 索引）
vocab = sorted(set(w for s in sentences for w in s))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)
print(f"词典大小 V={V}")
print("词典:", vocab)

## 2. Skip-gram 训练样本生成

Skip-gram 的规则：对句子里的每个词（中心词），把窗口内的词都当作它的正样本。

窗口大小 `window=1` 时：
```
句子 [猫, 吃, 鱼]
中心词 "吃" → 正样本 ("吃", "猫"),  ("吃", "鱼")
```

In [ ]:
def build_pairs(sentences, word2idx, window=1):
    """生成 (中心词idx, 上下文词idx) 正样本对。"""
    pairs = []
    for sent in sentences:
        idxs = [word2idx[w] for w in sent]
        for i, center in enumerate(idxs):
            for j in range(max(0, i - window), min(len(idxs), i + window + 1)):
                if j != i:
                    pairs.append((center, idxs[j]))
    return pairs

pairs = build_pairs(sentences, word2idx, window=1)
print(f"生成 {len(pairs)} 个正样本对")
print("前 5 个:", [(idx2word[c], idx2word[ctx]) for c, ctx in pairs[:5]])

## 3. 初始化词向量矩阵

两个矩阵：
- `W`：中心词矩阵，形状 (V, d)
- `W_ctx`：上下文词矩阵，形状 (V, d)

词向量维度 `d=8`（真实任务用 100～300，这里用小值加快演示）。

In [ ]:
d = 8  # 词向量维度

# 小随机数初始化
W     = np.random.randn(V, d) * 0.01
W_ctx = np.random.randn(V, d) * 0.01

print(f"W 形状: {W.shape}，W_ctx 形状: {W_ctx.shape}")
assert W.shape == (V, d)
assert W_ctx.shape == (V, d)

## 4. Sigmoid 函数

负采样目标函数需要把点积分数压缩到 (0, 1) 之间代表概率：

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

> **$e$ 和 sigmoid 是什么？**
>
> $e \approx 2.718$，是数学常数（初中不要求记住它的值）。
> $e^x$ 永远大于 0，随 $x$ 增大而增大。
> Sigmoid 用 $e^x$ 是为了：① 保证概率 $> 0$；② 让大分数的词获得更高概率。

In [ ]:
def sigmoid(x):
    # 数值稳定版：避免 exp(-x) 溢出
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))

assert abs(sigmoid(0) - 0.5) < 1e-9, "sigmoid(0) 应为 0.5"
assert sigmoid(100) > 0.999
assert sigmoid(-100) < 0.001
print(f"sigmoid(0)={sigmoid(0):.3f}, sigmoid(2)={sigmoid(2):.3f}, sigmoid(-2)={sigmoid(-2):.3f}")

## 5. 负采样

每次训练，对一个正样本 $(w_t, w_{pos})$，随机抽 $k$ 个负样本。

按词频的 3/4 次方采样：常用词被采到的概率适当降低。

In [ ]:
freq = np.zeros(V)
for sent in sentences:
    for w in sent:
        freq[word2idx[w]] += 1

neg_weights = freq ** 0.75
neg_weights /= neg_weights.sum()

def sample_negatives(center_idx, pos_idx, k, weights):
    """为 (center, pos) 抖取 k 个负样本。"""
    negs = []
    while len(negs) < k:
        candidate = np.random.choice(V, p=weights)
        if candidate != pos_idx and candidate != center_idx:
            negs.append(candidate)
    return negs

negs = sample_negatives(0, 1, k=3, weights=neg_weights)
assert 0 not in negs and 1 not in negs, "负样本不应包含中心词或正样本"
assert len(negs) == 3
print(f"示例负样本: {[idx2word[i] for i in negs]}")

## 6. 单步梯度更新

负采样目标：让正样本对的 sigmoid 趋近 1，负样本对趋近 0。

$$\mathcal{L} = -\log\sigma(v_c \cdot v_{pos}) - \sum_{i=1}^k \log\sigma(-v_c \cdot v_{neg_i})$$

In [ ]:
def train_step(center_idx, pos_idx, neg_idxs, W, W_ctx, lr=0.05):
    v_c   = W[center_idx]
    v_pos = W_ctx[pos_idx]

    score_pos = sigmoid(np.dot(v_c, v_pos))
    err_pos = score_pos - 1.0
    grad_center = err_pos * v_pos
    W_ctx[pos_idx] -= lr * err_pos * v_c

    for neg_idx in neg_idxs:
        v_neg = W_ctx[neg_idx]
        err_neg = sigmoid(np.dot(v_c, v_neg))
        grad_center += err_neg * v_neg
        W_ctx[neg_idx] -= lr * err_neg * v_c

    W[center_idx] -= lr * grad_center
    return -np.log(score_pos + 1e-9)

## 7. 训练循环

In [ ]:
EPOCHS = 500
K_NEG  = 5
LR     = 0.05

for epoch in range(EPOCHS):
    np.random.shuffle(pairs)
    total_loss = 0.0
    for center_idx, pos_idx in pairs:
        neg_idxs = sample_negatives(center_idx, pos_idx, K_NEG, neg_weights)
        total_loss += train_step(center_idx, pos_idx, neg_idxs, W, W_ctx, lr=LR)
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d}  avg_loss={total_loss/len(pairs):.4f}")

## 8. 验证：语义相近的词应该靠得更近

用**余弦相似度**衡量两个词向量的方向相似程度：

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

取值范围 $[-1, 1]$。1 表示完全同向（最相似），-1 表示完全反向。

> **验证假设**：
> 语料里「猫」「狗」「鸟」经常一起出现，所以训练后 `sim(猫, 狗)` 应大于 `sim(猫, 吃)`。

In [ ]:
def cosine_similarity(a, b):
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a < 1e-9 or norm_b < 1e-9:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))

def sim(w1, w2):
    return cosine_similarity(W[word2idx[w1]], W[word2idx[w2]])

# 余弦相似度数学性质验证
for w in vocab:
    assert abs(cosine_similarity(W[word2idx[w]], W[word2idx[w]]) - 1.0) < 1e-6
for w1 in vocab:
    for w2 in vocab:
        cs = cosine_similarity(W[word2idx[w1]], W[word2idx[w2]])
        assert -1.0 - 1e-6 <= cs <= 1.0 + 1e-6

print("所有词对的余弦相似度均在 [-1, 1] 内 ✓")
print()
print(f"  sim(猫,狗)  = {sim('猫','狗'):+.4f}   ← 同为动物")
print(f"  sim(猫,鸟)  = {sim('猫','鸟'):+.4f}")
print(f"  sim(鱼,肉)  = {sim('鱼','肉'):+.4f}   ← 同为食物")
print(f"  sim(猫,鱼)  = {sim('猫','鱼'):+.4f}   ← 猫经常吃鱼")
print(f"  sim(猫,米)  = {sim('猫','米'):+.4f}   ← 猫不吃米")

In [ ]:
# 核心断言：动物词之间应比动物-通用动词更相似
sim_cats_dogs = sim('猫', '狗')
sim_cats_eat  = sim('猫', '吃')

print(f"sim(猫,狗)={sim_cats_dogs:+.4f}  vs  sim(猫,吃)={sim_cats_eat:+.4f}")
assert sim_cats_dogs > sim_cats_eat, (
    f"期望 sim(猫,狗) > sim(猫,吃)，实际 {sim_cats_dogs:.4f} <= {sim_cats_eat:.4f}"
)
print("语义验证通过：动物词之间比动物-动词更相似 ✓")

## 9. 找最近邻

In [ ]:
def most_similar(word, top_k=3):
    idx = word2idx[word]
    scores = [(cosine_similarity(W[idx], W[j]), idx2word[j])
              for j in range(V) if j != idx]
    scores.sort(reverse=True)
    return scores[:top_k]

for test_word in ['猫', '鱼', '吃']:
    neighbors = most_similar(test_word, top_k=3)
    print(f"  与「{test_word}」最相似: {[f'{w}({s:+.3f})' for s, w in neighbors]}")

## 10. 小结

| 概念 | 作用 |
|------|------|
| 词向量矩阵 $W$ | 把每个词映射为 $d$ 维稠密向量 |
| Skip-gram | 用中心词预测周围词，定义了「邻居」关系 |
| 负采样 | 把全词典 softmax 简化成二分类，速度提升 10 倍+ |
| 余弦相似度 | 衡量两个词在语义空间中的方向相似程度 |

**下一站**：Transformer 的注意力机制让每个词动态计算自己的「邻居权重」，打破一词一向量的局限。

In [ ]:
# 保存训练好的词向量供集成测试使用（nbconvert cwd = notebook 目录）
import pickle
np.save('word_vectors.npy', W)
with open('word2idx.pkl', 'wb') as _f:
    pickle.dump(word2idx, _f)
print('词向量已保存: word_vectors.npy, word2idx.pkl')